# ESAT CPU vs GPU Speed Comparison

Compare official CPU (Rust) path vs GPU batched path on M4 Pro Metal.

**Note on Rust**: The official ESAT already rewrote core NMF loops in Rust + PyO3.
The CPU baseline here is CPU-bound Rust single-solve, not pure Python.
GPU acceleration comes from **batched** multi-model training via 
and DISP's delta-optimised .

In [ ]:
import os, sys, time, json, warnings, logging
import numpy as np
warnings.filterwarnings("ignore"); logging.disable(logging.CRITICAL)
sys.path.insert(0, os.path.abspath(".."))
from esat.model.batch_sa import BatchSA
from esat.error.bootstrap import Bootstrap
from esat.error.displacement import Displacement


def format_backend_label(backend):
    """Return a reader-facing label for the backend actually used."""
    backend = (backend or "unknown").lower()
    labels = {
        "metal": "GPU (Metal)",
        "cuda": "GPU (CUDA)",
        "gpu": "GPU",
        "cpu": "GPU request -> CPU fallback",
        "unknown": "GPU backend unknown",
    }
    return labels.get(backend, f"GPU ({backend})")


def q_range(values):
    return f"[{min(values):.0f}, {max(values):.0f}]"


def q_sanity(values, reference_q, multiplier=10.0):
    """Flag bootstrap/DISP Q ranges that are implausibly far above the base model."""
    values = np.asarray(values, dtype=float)
    if not np.all(np.isfinite(values)):
        return "INVALID_NONFINITE_Q"
    if reference_q > 0 and np.max(values) > reference_q * multiplier:
        return f"SUSPECT_DIVERGENCE_GT_{multiplier:.0f}X_BASE_Q"
    return "OK"


## Dataset — Baton Rouge

In [ ]:
from esat.data.datahandler import DataHandler

data_dir = os.path.join(os.getcwd(), "..", "data")
dh = DataHandler(
    input_path=os.path.join(data_dir, "Dataset-BatonRouge-con.csv"),
    uncertainty_path=os.path.join(data_dir, "Dataset-BatonRouge-unc.csv"),
    index_col="Date"
)
V, U = dh.get_data()
print(f"[DATASET] name=Baton Rouge")
print(f"[DATASET] V shape={V.shape}; samples={V.shape[0]}; features={len(dh.features)}")
print(f"[DATASET] factors=6; models=6; max_iter=2000; seed=42")


---
## 1. Base Model Training

CPU and GPU runs are separated so their timing and backend logs are easy to read.

In [ ]:
FACTORS, MODELS, MAX_ITER, SEED = 6, 6, 2000, 42
print(f"[BASE][SETUP] factors={FACTORS}; models={MODELS}; max_iter={MAX_ITER}; seed={SEED}")


### 1A. CPU baseline: official Rust single-solve path

In [ ]:
print("[BASE][CPU] Starting official Rust CPU baseline: BatchSA(use_gpu=False, parallel=False)")
t0 = time.time()
bsa_cpu = BatchSA(V=V, U=U, factors=FACTORS, models=MODELS, method="ls-nmf",
                  max_iter=MAX_ITER, seed=SEED, verbose=False,
                  parallel=False, use_gpu=False)
ok_cpu, _ = bsa_cpu.train()
t_cpu = time.time() - t0
sa_cpu = bsa_cpu.results[bsa_cpu.best_model]

print(f"[BASE][CPU] success={ok_cpu}")
print(f"[BASE][CPU] time_seconds={t_cpu:.3f}")
print(f"[BASE][CPU] best_model_1based={bsa_cpu.best_model + 1}")
print(f"[BASE][CPU] q_true={sa_cpu.Qtrue:.0f}")


### 1B. GPU requested: batched LS-NMF path

In [ ]:
print("[BASE][GPU] Starting batched path: BatchSA(use_gpu=True, parallel=False)")
t0 = time.time()
bsa_gpu = BatchSA(V=V, U=U, factors=FACTORS, models=MODELS, method="ls-nmf",
                  max_iter=MAX_ITER, seed=SEED, verbose=False,
                  use_gpu=True, parallel=False)
ok_gpu, _ = bsa_gpu.train()
t_gpu = time.time() - t0
sa_gpu = bsa_gpu.results[bsa_gpu.best_model]
base_gpu_backend = sa_gpu.metadata.get("backend", "unknown")
base_gpu_label = format_backend_label(base_gpu_backend)

print(f"[BASE][GPU] success={ok_gpu}")
print(f"[BASE][GPU] backend={base_gpu_backend} ({base_gpu_label})")
print(f"[BASE][GPU] time_seconds={t_gpu:.3f}")
print(f"[BASE][GPU] best_model_1based={bsa_gpu.best_model + 1}")
print(f"[BASE][GPU] q_true={sa_gpu.Qtrue:.0f}")


### 1C. Base model comparison

In [ ]:
print(f"[BASE][COMPARE] Base model training: {MODELS} models x {MAX_ITER} iterations")
print(f"{'Group':>30s} {'Time':>10s} {'Q(true)':>12s} {'Best':>6s}")
print(f"{'CPU (Rust)':>30s} {t_cpu:>9.3f}s {sa_cpu.Qtrue:>11.0f} {bsa_cpu.best_model + 1:>6d}")
print(f"{base_gpu_label:>30s} {t_gpu:>9.3f}s {sa_gpu.Qtrue:>11.0f} {bsa_gpu.best_model + 1:>6d}")
if t_gpu > 0:
    print(f"[BASE][COMPARE] speedup_cpu_over_gpu={t_cpu / t_gpu:.1f}x")


---
## 2. Bootstrap

CPU and GPU bootstrap runs are separated. The GPU cell reports the backend returned by the batched kernel.

In [ ]:
sa_cpu.metadata["converge_delta"] = 0.1
sa_cpu.metadata["converge_n"] = 100
sa_gpu.metadata["converge_delta"] = 0.1
sa_gpu.metadata["converge_n"] = 100

BS_N, BS_BLOCK, BS_THRESH = 20, 10, 0.6
print(f"[BOOTSTRAP][SETUP] runs={BS_N}; block_size={BS_BLOCK}; threshold={BS_THRESH}")


### 2A. CPU Bootstrap

In [ ]:
print("[BOOTSTRAP][CPU] Starting Bootstrap(use_gpu=False, parallel=False)")
t0 = time.time()
bs_cpu = Bootstrap(sa=sa_cpu, feature_labels=dh.features,
                   bootstrap_n=BS_N, block_size=BS_BLOCK, threshold=BS_THRESH,
                   parallel=False, use_gpu=False)
bs_cpu.run(keep_H=True, block=True)
t_bs_cpu = time.time() - t0
q_cpu = [bs_cpu.bs_results[k]["model"].Qtrue for k in bs_cpu.bs_results]

print(f"[BOOTSTRAP][CPU] time_seconds={t_bs_cpu:.3f}")
print(f"[BOOTSTRAP][CPU] seconds_per_run={t_bs_cpu / BS_N:.4f}")
print(f"[BOOTSTRAP][CPU] q_true_range={q_range(q_cpu)}")
print(f"[BOOTSTRAP][CPU] q_true_sanity={q_sanity(q_cpu, sa_cpu.Qtrue)}")


### 2B. GPU requested Bootstrap

In [ ]:
print("[BOOTSTRAP][GPU] Starting Bootstrap(use_gpu=True, parallel=False)")
t0 = time.time()
bs_gpu = Bootstrap(sa=sa_gpu, feature_labels=dh.features,
                   bootstrap_n=BS_N, block_size=BS_BLOCK, threshold=BS_THRESH,
                   parallel=False, use_gpu=True)
bs_gpu.run(keep_H=True, block=True)
t_bs_gpu = time.time() - t0
q_gpu = [bs_gpu.bs_results[k]["model"].Qtrue for k in bs_gpu.bs_results]
bootstrap_gpu_backend = bs_gpu.metadata.get("backend", "unknown")
bootstrap_gpu_label = format_backend_label(bootstrap_gpu_backend)

print(f"[BOOTSTRAP][GPU] backend={bootstrap_gpu_backend} ({bootstrap_gpu_label})")
print(f"[BOOTSTRAP][GPU] time_seconds={t_bs_gpu:.3f}")
print(f"[BOOTSTRAP][GPU] seconds_per_run={t_bs_gpu / BS_N:.4f}")
print(f"[BOOTSTRAP][GPU] q_true_range={q_range(q_gpu)}")
print(f"[BOOTSTRAP][GPU] q_true_sanity={q_sanity(q_gpu, sa_gpu.Qtrue)}")


### 2C. Bootstrap comparison

In [ ]:
print(f"[BOOTSTRAP][COMPARE] Bootstrap: {BS_N} runs")
print(f"{'Group':>30s} {'Time':>10s} {'/run':>10s} {'Q range':>18s} {'Q sanity':>28s}")
print(f"{'CPU (Rust)':>30s} {t_bs_cpu:>9.3f}s {t_bs_cpu / BS_N:>9.4f}s {q_range(q_cpu):>18s} {q_sanity(q_cpu, sa_cpu.Qtrue):>28s}")
print(f"{bootstrap_gpu_label:>30s} {t_bs_gpu:>9.3f}s {t_bs_gpu / BS_N:>9.4f}s {q_range(q_gpu):>18s} {q_sanity(q_gpu, sa_gpu.Qtrue):>28s}")
if t_bs_gpu > 0:
    print(f"[BOOTSTRAP][COMPARE] speedup_cpu_over_gpu={t_bs_cpu / t_bs_gpu:.1f}x")


---
## 3. DISP

DISP is slower than base training and bootstrap. These cells are separated so CPU and GPU runs can be skipped or rerun independently.

In [ ]:
MAX_SEARCH, THRESH_DQ = 20, 0.1
print(f"[DISP][SETUP] factors={FACTORS}; features={len(dh.features)}; max_search={MAX_SEARCH}; threshold_dQ={THRESH_DQ}")


### 3A. CPU DISP

In [ ]:
print("[DISP][CPU] Starting Displacement(use_gpu=False, parallel=False)")
t0 = time.time()
disp_cpu = Displacement(sa=sa_cpu, feature_labels=dh.features,
                        max_search=MAX_SEARCH, threshold_dQ=THRESH_DQ,
                        parallel=False, use_gpu=False)
disp_cpu.run()
t_disp_cpu = time.time() - t0

print(f"[DISP][CPU] time_seconds={t_disp_cpu:.3f}")
print(f"[DISP][CPU] increase_factor_count={len(disp_cpu.increase_results)}")
print(f"[DISP][CPU] decrease_factor_count={len(disp_cpu.decrease_results)}")


### 3B. GPU requested DISP

In [ ]:
print("[DISP][GPU] Starting Displacement(use_gpu=True, parallel=False)")
t0 = time.time()
disp_gpu = Displacement(sa=sa_gpu, feature_labels=dh.features,
                        max_search=MAX_SEARCH, threshold_dQ=THRESH_DQ,
                        parallel=False, use_gpu=True)
disp_gpu.run()
t_disp_gpu = time.time() - t0
disp_gpu_backend = getattr(disp_gpu, "_backend", "unknown")
disp_gpu_label = format_backend_label(disp_gpu_backend)

print(f"[DISP][GPU] backend={disp_gpu_backend} ({disp_gpu_label})")
print(f"[DISP][GPU] time_seconds={t_disp_gpu:.3f}")
print(f"[DISP][GPU] increase_factor_count={len(disp_gpu.increase_results)}")
print(f"[DISP][GPU] decrease_factor_count={len(disp_gpu.decrease_results)}")


### 3C. DISP comparison

In [ ]:
print(f"[DISP][COMPARE] DISP: {FACTORS} factors x {len(dh.features)} features, max_search={MAX_SEARCH}")
print(f"{'Group':>30s} {'Time':>10s} {'inc factors':>13s} {'dec factors':>13s}")
print(f"{'CPU (Rust)':>30s} {t_disp_cpu:>9.3f}s {len(disp_cpu.increase_results):>13d} {len(disp_cpu.decrease_results):>13d}")
print(f"{disp_gpu_label:>30s} {t_disp_gpu:>9.3f}s {len(disp_gpu.increase_results):>13d} {len(disp_gpu.decrease_results):>13d}")
if t_disp_gpu > 0:
    print(f"[DISP][COMPARE] speedup_cpu_over_gpu={t_disp_cpu / t_disp_gpu:.1f}x")


---
## Comparison Summary

In [ ]:
summary_gpu_label = "GPU requested"
print("=" * 82)
print(f"{'Stage':25s} {'CPU (Rust)':>12s} {summary_gpu_label:>18s} {'Speedup':>10s} {'GPU backend':>18s}")
print("-" * 82)
print(f"{'Base model':25s} {t_cpu:>9.3f}s {t_gpu:>15.3f}s {t_cpu / t_gpu:>9.1f}x {base_gpu_backend:>18s}")
print(f"{'Bootstrap':25s} {t_bs_cpu:>9.3f}s {t_bs_gpu:>15.3f}s {t_bs_cpu / t_bs_gpu:>9.1f}x {bootstrap_gpu_backend:>18s}")
print(f"{'DISP':25s} {t_disp_cpu:>9.3f}s {t_disp_gpu:>15.3f}s {t_disp_cpu / t_disp_gpu:>9.1f}x {disp_gpu_backend:>18s}")
print("-" * 82)
total_cpu = t_cpu + t_bs_cpu + t_disp_cpu
total_gpu = t_gpu + t_bs_gpu + t_disp_gpu
print(f"{'Total':25s} {total_cpu:>9.3f}s {total_gpu:>15.3f}s {total_cpu / total_gpu:>9.1f}x {'mixed':>18s}")
print("=" * 82)
print(f"[SUMMARY] hardware=M4 Pro MacBook")
print(f"[SUMMARY] dataset=Baton Rouge; samples={V.shape[0]}; features={V.shape[1]}")
print(f"[SUMMARY] backend_by_stage: base={base_gpu_backend}; bootstrap={bootstrap_gpu_backend}; disp={disp_gpu_backend}")


## Key Takeaways

1. **Base model** — GPU batched training packs all N models into a single 3D call.
2. **Bootstrap** — Same batching across all B resamples. Speedup grows with B.
3. **DISP** — Two independent optimisations:
   - Delta-optimised  (doesn't recompute full WH for single H changes)
   - Batched SA.train for the refit step after each dQ is found
4. The official Rust single-solve is already a significant improvement over the original
   pure-Python NMF loop. GPU batching adds further gains when multiple models need
   training (Bootstrap, DISP refits, multi-seed batch SA).